In [ ]:
!pip install torchcfm torchdyn torchsde torchdiffeq

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import time
import copy
import matplotlib.pyplot as plt
import torch
import torchdiffeq
import torchsde
from torchdyn.core import NeuralODE
from torchvision import datasets, transforms
from torchvision.transforms import ToPILImage
from torchvision.utils import save_image
from tqdm import tqdm


from torchcfm.conditional_flow_matching import *
from torchcfm.models.unet.unet import UNetModelWrapper

data_root = "/kaggle/working/data" if os.path.exists("/kaggle") else "../data"
savedir = "/kaggle/working/models/cifar10_cfm" if os.path.exists("/kaggle") else "models/cifar10_cfm"
os.makedirs(savedir, exist_ok=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
hp = {
    "batch_size": 64,
    'sigma': 0.0,
    'warmup_steps': 5000,
    "total_steps": 1_000,
"sample_every": 500,
"save_every": 500,

}

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
trainset = datasets.CIFAR10(
    "../data",
    train=True, 
    download=True
    ,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)), transforms.RandomHorizontalFlip()])
)

dataloader = torch.utils.data.DataLoader(
    trainset, batch_size=hp['batch_size'], shuffle=True, drop_last=True,num_workers=2,
    pin_memory=(device.type == "cuda"),
)

100%|██████████| 170M/170M [00:07<00:00, 21.8MB/s] 


In [17]:
def warmup_lr(step):
    return min(step, hp['warmup_steps']) / hp['warmup_steps']

def generate_samples(model, parallel, savedir, step, net_="normal"):
    """Save 64 generated images (8 x 8) for sanity check along training.

    Parameters
    ----------
    model:
        represents the neural network that we want to generate samples from
    parallel: bool
        represents the parallel training flag. Torchdyn only runs on 1 GPU, we need to send the models from several GPUs to 1 GPU.
    savedir: str
        represents the path where we want to save the generated images
    step: int
        represents the current step of training
    """
    model.eval()

    model_ = copy.deepcopy(model)
    if parallel:
        # Send the models from GPU to CPU for inference with NeuralODE from Torchdyn
        model_ = model_.module.to(device)

    node_ = NeuralODE(model_, solver="euler", sensitivity="adjoint")
    with torch.no_grad():
        traj = node_.trajectory(
            torch.randn(64, 3, 32, 32, device=device),
            t_span=torch.linspace(0, 1, 100, device=device),
        )
        traj = traj[-1, :].view([-1, 3, 32, 32]).clip(-1, 1)
        traj = traj / 2 + 0.5
    save_image(traj, os.path.join(savedir, f"{net_}_generated_FM_images_step_{step}.png"), nrow=8)

    model.train()


def ema(source, target, decay):
    source_dict = source.state_dict()
    target_dict = target.state_dict()
    for key in source_dict.keys():
        target_dict[key].data.copy_(
            target_dict[key].data * decay + source_dict[key].data * (1 - decay)
        )


In [18]:
net_model = UNetModelWrapper(
    dim=(3,32,32),
    num_res_blocks=2,
    num_channels=128,
    channel_mult=[1,2,2,2],
    num_heads=4,
    num_head_channels=64, 
    attention_resolutions="16",
    dropout=0.1).to(device)

ema_model = copy.deepcopy(net_model)
optim = torch.optim.AdamW(net_model.parameters(), lr=2e-4)
scheduler = torch.optim.lr_scheduler.LambdaLR(optim, warmup_lr)

# Model size 
model_size = 0
for param in net_model.parameters():
    model_size += param.data.nelement()
print(f"Model params: {model_size / 1024 /1024}")

FM = ExactOptimalTransportConditionalFlowMatcher(sigma=0.0)

Model params: 34.09033489227295


In [21]:
data_iter = iter(dataloader)
pbar = tqdm(range(hp["total_steps"]))
for step in pbar:
    optim.zero_grad()
    x1, _ = next(data_iter)
    x1 = x1.to(device)
    x0 = torch.randn_like(x1)

    t, xt, ut = FM.sample_location_and_conditional_flow(x0, x1)
    t = t.to(device)
    xt = xt.to(device)
    ut = ut.to(device)

    vt = net_model(t, xt)
    loss = torch.mean((ut -vt)**2)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(net_model.parameters(), 1.0)
    optim.step()
    scheduler.step()
    ema(net_model, ema_model, 0.9999)
    pbar.set_postfix(loss=loss.item(), lr=scheduler.get_last_lr()[0])

    if step % hp["sample_every"] == 0:
        generate_samples(net_model, False, savedir, step, net_="normal")
        generate_samples(ema_model, False, savedir, step, net_="ema")

    if step % hp["save_every"] == 0:
        torch.save(
            {
                "net_model": net_model.state_dict(),
                "ema_model": ema_model.state_dict(),
                "sched": scheduler.state_dict(),
                "optim": optim.state_dict(),
                "step": step,
            },
            os.path.join(savedir, f"cifar10_weights_step_{step}.pt"),
        )

KeyboardInterrupt: 